# 13.02 · Speech-to-text with Groq Whisper: voicemails to tickets

**Module 13 · Multimodal & Voice** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lesson 13.01 and Lesson 02.03 (structured output). Read the module README first. |
| 🔑 Keys | optional: `GROQ_API_KEY` (Whisper), `GOOGLE_API_KEY` (Gemini TTS makes the test recordings, free tier) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** sample rate, channels and bit depth, and compute whether a recording fits Groq's upload limit.
2. **Transcribe** audio with Groq Whisper (`verbose_json`) and measure word error rate (WER) against a known script.
3. **Improve** names and IDs with a spelling-hint `prompt`, and know its 224-token limit.
4. **Choose** between transcribing Spanish as Spanish and translating it to English (`whisper-large-v3` only).
5. **Detect** suspicious output with segment confidence fields, including Whisper's text-over-silence hallucinations.
6. **Chunk** long recordings with overlap and stitch the timestamps, then **turn** voicemails into routed support tickets.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Groq speech to text](https://console.groq.com/docs/speech-to-text) | Models, prices, file limits, `prompt`, `verbose_json`, timestamps |
| [Whisper Large V3 Turbo](https://console.groq.com/docs/model/whisper-large-v3-turbo) | The default: fast, $0.04 per hour, no translation |
| [Groq rate limits](https://console.groq.com/docs/rate-limits) | Free tier: 20 requests/min, 2,000/day, 7,200 audio seconds/hour |
| [Gemini speech generation](https://ai.google.dev/gemini-api/docs/speech-generation) | How the test recordings are made (Interactions API, 16 kHz WAV) |
| [Whisper paper, section 4](https://arxiv.org/abs/2212.04356) | Why WER needs text normalization before you compare systems |

> **Why this matters at work.** Voicemails, sales calls, interviews and meetings are recorded every day and almost never read. Transcription makes them searchable and lets an agent act on them, but only if you know how wrong the transcript is and where. This lesson turns support voicemails for Orbita Cloud into tickets, measures the errors, and sends the uncertain ones to a person.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "groq>=0.37.1,<1" "google-genai>=2.25.0,<3" "numpy>=2.3.0,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-groq", "groq", "google-genai", "numpy",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/13-multimodal-and-voice/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'GOOGLE_API_KEY')
LANGSMITH_PROJECT = 'm13-speech-to-text'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_GOOGLE_KEY = is_set("GOOGLE_API_KEY") or is_set("GEMINI_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m13-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("vision    :", settings.model_vision, "(Groq, preview)")
print("multimodal:", settings.model_multimodal)
print("stt / tts :", settings.model_stt, "/", settings.tts_provider)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_GOOGLE_KEY:
    print("⚠️ No GOOGLE_API_KEY: 💸 Gemini cells will print ⏭️ Skipped (free key: https://aistudio.google.com/apikey).")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models (qwen3.8 vision, Orpheus) warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import json
import time

from IPython.display import Audio, display

SAMPLES = REPO_ROOT / "data" / "samples" / "multimodal"
DATA = json.loads((SAMPLES / "voicemails.json").read_text(encoding="utf-8"))
VOICEMAILS = {vm["id"]: vm for vm in DATA["voicemails"]}
AUDIO_DIR = REPO_ROOT / "data" / "outputs" / "m13" / "audio"   # git-ignored: generated recordings live here
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print(len(VOICEMAILS), "voicemail scripts:", ", ".join(f"{k} ({v['language']})" for k, v in VOICEMAILS.items()))

**You should now see** `4 voicemail scripts: vm-01 (en), vm-02 (en), vm-03 (es), vm-04 (en)`.

The scripts are fictional customers calling Orbita Cloud support. Because you know exactly what each recording says, you can **measure** the transcript instead of eyeballing it.

---
## Part 1 · What an audio file is, and what fits in one request

A recording is a list of numbers: **samples** per second (sample rate), **channels** (mono or stereo) and **bits** per sample. Uncompressed size = rate × channels × bits ÷ 8 per second. Whisper works on 16 kHz mono internally, so anything more is upload you pay for in time, not in accuracy.

In [ ]:
# 🧪 Offline: sizes, and how many minutes fit under Groq's 25 MB free-tier limit.
from mastery.voice import tone, wav_size_bytes

MB = 1024 * 1024
rows = []
for label, rate, ch in (("phone (8 kHz mono)", 8_000, 1), ("speech (16 kHz mono)", 16_000, 1),
                        ("Gemini TTS (24 kHz mono)", 24_000, 1), ("CD (44.1 kHz stereo)", 44_100, 2)):
    per_min = wav_size_bytes(60, rate, ch) / MB
    rows.append({"format": label, "MB per minute": round(per_min, 2), "minutes in 25 MB": int(25 / per_min)})
table(rows, ["format", "MB per minute", "minutes in 25 MB"])

beep = tone(440, 1.0)
print(f"\n1 s tone: {beep.rate} Hz, {len(beep.samples)} samples, {beep.rms_db():.1f} dBFS")
display(Audio(beep.to_wav(), rate=beep.rate))

**You should now see** 16 kHz mono at about 1.83 MB per minute (13 minutes in 25 MB) and CD quality at about 10 MB per minute (2 minutes), then a one-second tone with a player.

### What just happened?

Convert to **16 kHz mono** before uploading: a CD-quality hour-long meeting is 600 MB, the same audio at 16 kHz mono is 110 MB, and as FLAC (lossless, made with `ffmpeg`) about half of that. Anything still over 25 MB gets **chunked** (Part 7).

---
## Part 2 · Make the test recordings (Gemini TTS, free tier)

Real customer voicemails are personal data. To practise, you **synthesize** the four scripts with Gemini TTS, directly at 16 kHz, and keep them in `data/outputs/m13/audio/` (git-ignored). Later runs read the files from there and send nothing.

In [ ]:
# 💸 Gemini TTS: one request per voicemail the first time; read from disk afterwards.
from mastery.voice import cached_audio, gemini_tts, read_wav

clips = {}
for vm_id, vm in VOICEMAILS.items():
    path = AUDIO_DIR / f"{vm_id}.wav"
    if path.is_file() or HAVE_GOOGLE_KEY:
        data, from_cache = cached_audio(path, lambda vm=vm: gemini_tts(vm["script"], voice=vm["voice"], style=vm["style"], sample_rate=16_000))
        clips[vm_id] = data
        clip = read_wav(data)
        print(f"{vm_id}: {clip.seconds:4.1f} s, {clip.rate} Hz, {len(data) / 1024:4.0f} KB", "(from disk)" if from_cache else "(synthesized)")
if not clips:
    print("⏭️ Skipped: GOOGLE_API_KEY is not set and no recordings exist yet. The lesson uses SIMULATED transcripts.")
elif "vm-01" in clips:
    display(Audio(clips["vm-01"], rate=16_000))

> 💸 Calls Gemini TTS (`gemini-3.8-flash-lite-tts`, free tier) four times, once. Free-tier content may be used by Google to improve its products; these scripts are synthetic. Without `GOOGLE_API_KEY` (and no saved files) it prints `⏭️ Skipped`.

**You should now see** four lines like `vm-01: 12.5 s, 16000 Hz, 390 KB (synthesized)` (`(from disk)` on later runs) and a player: listen to it. Each voice has its own `style`, so vm-04 sounds rushed.

---
## Part 3 · First transcription and word error rate

`langchain-groq` has no speech-to-text class, so `GroqTranscriber` calls the `groq` SDK directly with `response_format="verbose_json"`. Without a key or recordings, `FakeTranscriber` returns **simulated** transcripts from `voicemails.json` (the scripts with the errors Whisper typically makes), so every later cell still runs.

**WER** = (substituted + deleted + inserted words) ÷ words in the script. `wer()` lowercases and drops punctuation first, otherwise "C-1002," and "c 1002" would count as an error.

In [ ]:
# Choose the transcriber once: real Groq Whisper (LIVE) or the simulated stand-in.
from mastery.voice import FakeTranscriber, GroqTranscriber, wer, whisper_cost

LIVE = HAVE_GROQ_KEY and len(clips) == len(VOICEMAILS)
if LIVE:
    stt = GroqTranscriber()            # settings.model_stt: whisper-large-v3-turbo
else:
    stt = FakeTranscriber({k: v["simulated_whisper"] for k, v in VOICEMAILS.items()})
MODE = "live" if LIVE else "SIMULATED"
print("transcriber:", stt.model, f"({MODE})")
if not LIVE:
    print("⏭️ Skipped: needs GROQ_API_KEY and the Part 2 recordings (see the setup cell).")

In [ ]:
# 💸 (LIVE) Transcribe all four voicemails and score them against the scripts.
transcripts, rows = {}, []
for vm_id, vm in VOICEMAILS.items():
    start = time.perf_counter()
    t = stt.transcribe(clips.get(vm_id), key=vm_id, language=vm["language"])
    seconds = time.perf_counter() - start
    transcripts[vm_id] = t
    rows.append({"id": vm_id, "lang": vm["language"], "audio s": round(t.duration or 0, 1), "WER": wer(vm["script"], t.text),
                 "request s": round(seconds, 2), "cost $": f"{whisper_cost(t.duration or 0):.5f}", "mode": MODE})
table(rows, ["id", "lang", "audio s", "WER", "request s", "cost $", "mode"])
print("\nvm-01 transcript:", transcripts["vm-01"].text)
print("vm-01 script    :", VOICEMAILS["vm-01"]["script"])

> 💸 With `LIVE`, four Whisper requests (about 50 seconds of audio, a fraction of a cent, well inside the free tier's 7,200 audio seconds per hour).

**You should now see** a table with WER per voicemail. Simulated: 0.07–0.14, with `Nimbus bro` for *Nimbus Pro* and `Gomez` for *Gómez* in vm-01. Live: usually below 0.1 on these clean synthetic voices, with **most remaining errors on names, IDs and number formats**, and each request well under a second.

### What just happened?

- WER treats "$49" vs "forty nine dollars" and "Gomez" vs "Gómez" as errors although a human would accept them. Normalize what you don't care about, and also look at the errors that **do** matter: a wrong customer ID routes the ticket to the wrong account.
- Each request is billed for at least **10 seconds**: thousands of 2-second clips cost five times their length. Batch short clips into one file when you can.

---
## Part 4 · Spelling hints with `prompt`

Whisper's `prompt` is text the model treats as if it came just before the audio. Put the product names and ID formats it can't know in it. It is **not** an instruction ("transcribe carefully" does nothing) and is limited to **224 tokens**.

In [ ]:
# 💸 (LIVE) The same four voicemails with a spelling-hint prompt.
HINTS = "Orbita Cloud support voicemail. Products: Orbita Cloud, Nimbus Pro. Customers: Andes Logistics. IDs like C-1002, C-2042."
print(f"prompt: {len(HINTS)} characters ≈ {len(HINTS) // 4} tokens (limit 224)\n")

def term_hits(text):
    return sum(term.casefold() in text.casefold() for term in DATA["hint_terms"])

rows, hinted = [], {}
for vm_id, vm in VOICEMAILS.items():
    hinted[vm_id] = stt.transcribe(clips.get(vm_id), key=vm_id, prompt=HINTS, language=vm["language"])
    rows.append({"id": vm_id, "WER plain": wer(vm["script"], transcripts[vm_id].text),
                 "WER hinted": wer(vm["script"], hinted[vm_id].text),
                 "terms plain": term_hits(transcripts[vm_id].text), "terms hinted": term_hits(hinted[vm_id].text),
                 "terms in script": term_hits(vm["script"]), "mode": MODE})
table(rows, ["id", "WER plain", "WER hinted", "terms plain", "terms hinted", "terms in script", "mode"])

> 💸 With `LIVE`, four more Whisper requests.

**You should now see** the hinted WER lower or equal on every row and `terms hinted` reaching `terms in script` (simulated: exactly; live: usually, since a clean synthetic voice is easier than a phone line).

### What just happened?

The prompt fixed the words it **contains** and nothing else. That makes it a per-domain asset: keep the hint list in config, next to your product catalogue, and measure it on recordings like Part 3's, never on one lucky clip. Too long, or full of words that aren't said, and Whisper may start writing them anyway.

---
## Part 5 · Spanish: transcribe or translate?

vm-03 is in Spanish. Two options: **transcribe** it as Spanish (turbo, with `language="es"`), or **translate** it to English text in one step with `whisper-large-v3` (the only Groq model that translates, $0.111 per hour). Translation is convenient for an English-speaking team, but you lose the customer's own words.

In [ ]:
# 💸 (LIVE) vm-03 transcribed as Spanish and translated to English.
vm = VOICEMAILS["vm-03"]
spanish = hinted["vm-03"]
if LIVE:
    english = GroqTranscriber("whisper-large-v3", translate=True).transcribe(clips["vm-03"])
else:
    english = FakeTranscriber({"vm-03": {"text": vm["translation"]}}).transcribe(key="vm-03")
print(f"transcribed ({MODE}):", spanish.text)
print(f"translated  ({MODE}):", english.text)
print("\ncost per hour: turbo $0.04, large-v3 $0.111 → translating every Spanish call costs ~2.8x")

> 💸 With `LIVE`, one `whisper-large-v3` translation request.

**You should now see** the Spanish transcript (*Quiero cambiar mi plan de Orbita Cloud a Nimbus Pro…*) and an English translation (*I want to change my Orbita Cloud plan to Nimbus Pro…*).

### What just happened?

For a ticket, keep **both**: the original transcript is the record (and what you quote back to the customer), and the English text is for the team. You can also translate the Spanish transcript later with the fast text model, which keeps the cheaper turbo for every call. Exercise 3 compares the two.

---
## Part 6 · Timestamps, confidence and hallucinations

`verbose_json` returns **segments** (start, end, text) with three fields worth reading: `avg_logprob` (confidence, closer to 0 is better), `no_speech_prob` (was it speech at all?) and `compression_ratio` (above about 2.4 the text repeats itself). Whisper is known to **write text over silence or noise**, often a polite sign-off learned from video subtitles.

In [ ]:
# 💸 (LIVE) Segments of vm-01, then 12 seconds of pure noise.
from mastery.voice import flag_segments, parse_verbose_json, silence, with_noise

t = transcripts["vm-01"]
table([{"start": s.start, "end": s.end, "avg_logprob": round(s.avg_logprob, 2), "no_speech": round(s.no_speech_prob, 2),
        "text": s.text[:60]} for s in t.segments], ["start", "end", "avg_logprob", "no_speech", "text"])
if t.words:
    print("\nfirst words:", [(w.get("word"), round(w.get("start", 0), 2)) for w in t.words[:6]])

noise = with_noise(silence(12), snr_db=0, seed=1)
if LIVE:
    noisy = stt.transcribe(noise)
else:
    noisy = parse_verbose_json(DATA["simulated_noise_only"])
print(f"\nnoise only ({MODE}) → {noisy.text!r}")
for seg, why in flag_segments(noisy) or [(None, "nothing flagged")]:
    print("   ⚠️", why, "→ drop this segment" if seg else "")

> 💸 With `LIVE`, one Whisper request on 12 seconds of noise (billed as 12 s).

**You should now see** vm-01's sentences with start/end times and `avg_logprob` around -0.2 to -0.4 (live) and, if Groq returned them, word timestamps. For the noise: simulated, *'Thank you for watching.'* flagged as `likely no speech`. Live, Whisper may return an empty string, a short phrase, or a flagged segment; record what you got in the research log.

### What just happened?

Never trust a transcript of audio you haven't checked for speech. Two cheap defences: skip clips whose loudness (`rms_db()`) is near silence **before** sending them (Lesson 13.05 does this properly with voice activity detection), and drop segments that `flag_segments` marks. Timestamps let a reviewer jump straight to the doubtful second instead of replaying the whole call.

---
## Part 7 · Long recordings: chunk with overlap

Over 25 MB (about 13 minutes of 16 kHz WAV), split the audio. Cut with an **overlap** so a word on a boundary is whole in one of the chunks, shift each chunk's timestamps by its start, and drop segments the previous chunk already covered. `transcribe_long` does this; here it runs on three voicemails joined into one 45-second "call", cut into 20-second chunks to see it work.

In [ ]:
# 💸 (LIVE) One long recording, transcribed in overlapping chunks.
from mastery.voice import chunk, concat, transcribe_long

if LIVE:
    long_call = concat(*[c for vm_id in ("vm-01", "vm-02", "vm-04") for c in (read_wav(clips[vm_id]).resample(16_000), silence(1.5))])
    long_stt = stt
else:
    long_call = concat(tone(300, 20), silence(16), tone(300, 20))   # stand-in signal, 56 s
    def scripted(key):   # SIMULATED: each 20 s chunk re-hears the previous chunk's last 2 s
        i = int(key.split("#")[1])
        segs = [{"start": 0.5, "end": 1.8, "text": f"(overlap of chunk {i - 1})"}] if i else []
        segs += [{"start": 2.0 if i else 0.0, "end": 17.5, "text": f"words of chunk {i}"}, {"start": 18.0, "end": 20.0, "text": f"end of chunk {i}"}]
        return {"text": "", "segments": segs}
    long_stt = FakeTranscriber(scripted)
print(f"{long_call.seconds:.1f} s → chunks start at", [s for s, _ in chunk(long_call, 20, 2)])
stitched = transcribe_long(long_call, long_stt, max_seconds=20, overlap_seconds=2, key="long")
for s in stitched.segments:
    print(f"  {s.start:6.1f}–{s.end:6.1f}  {s.text[:70]}")

> 💸 With `LIVE`, three Whisper requests (about 45 seconds of audio).

**You should now see** chunk starts `[0.0, 18.0, 36.0]` (simulated: a 56 s stand-in signal; live: the three joined voicemails, about 45 s), then segments with times that keep increasing across chunks and no sentence repeated from an overlap. Simulated, the `(overlap of chunk …)` lines are gone.

### What just happened?

Without the shift, every chunk's timestamps restart at 0; without the de-duplication, words in the overlap appear twice. In production, cut at **silences** instead of fixed times (Lesson 13.05) and use 10-minute chunks: the free tier allows 7,200 audio seconds per hour, so a 3-hour backlog has to be spread over the day.

---
## Part 8 · Real workflow: voicemail → ticket → queue

The transcript is only useful once it becomes work: who called, about what, how urgent, and how to call back. A fast text model fills a typed `VoicemailTicket` (Lesson 02.03). The offline twin uses keyword rules, which is what many teams start with.

In [ ]:
# 🧪 Offline: the ticket schema and a keyword-rule twin.
import re
from typing import Literal

from pydantic import BaseModel, Field

from mastery.multimodal import score_fields

class VoicemailTicket(BaseModel):
    customer_id: str | None = Field(description="Customer ID like C-1002, normalized with the hyphen")
    intent: Literal["outage", "billing", "plan_change", "access", "other"]
    urgency: Literal["low", "normal", "high"] = Field(description="high if the customer says urgent or service is down")
    callback_number: str | None = Field(default=None, description="Phone number as digits and spaces, if given")
    summary: str = Field(description="One sentence in English")

def rule_ticket(text):
    low = text.casefold()
    cid = re.search(r"c[\s-]?(\d{4})", low)
    phone = re.search(r"(\d{3})[\s-](\d{3})[\s-](\d{4})", text)
    intent = next((name for name, words in [("outage", ("error", "down", "caído")), ("billing", ("charged", "refund", "cobr")),
                                           ("plan_change", ("plan",)), ("access", ("locked", "password"))]
                   if any(w in low for w in words)), "other")
    return VoicemailTicket(customer_id=f"C-{cid.group(1)}" if cid else None, intent=intent,
                           urgency="high" if ("urgent" in low or intent == "outage") else "normal",
                           callback_number=" ".join(phone.groups()) if phone else None, summary=text[:80])

FIELDS = ["customer_id", "intent", "urgency", "callback_number"]
rows = []
for vm_id, vm in VOICEMAILS.items():
    scored = score_fields(rule_ticket(hinted[vm_id].text), vm["ticket"], FIELDS)
    rows.append({"id": vm_id, "method": "rules", "accuracy": scored["accuracy"], "wrong": "; ".join(scored["wrong"])[:70]})
table(rows, ["id", "method", "accuracy", "wrong"])

**You should now see** (simulated transcripts) 1.0 for all four voicemails: the rules work on these four. Rename "locked out" to "can't get in" and they fail, which is why the next cell uses a model.

In [ ]:
# 💸 The fast text model fills the ticket from each hinted transcript; high urgency pages on-call.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    ticketer = get_model("fast", temperature=0).with_structured_output(VoicemailTicket)
    rows = []
    for vm_id, vm in VOICEMAILS.items():
        ticket = ticketer.invoke("Create a support ticket from this voicemail transcript to Orbita Cloud.\n\n" + hinted[vm_id].text)
        scored = score_fields(ticket, vm["ticket"], FIELDS)
        queue = "📟 on-call" if ticket.urgency == "high" else "📥 support queue"
        rows.append({"id": vm_id, "intent": ticket.intent, "urgency": ticket.urgency, "queue": queue,
                     "accuracy": scored["accuracy"], "summary": ticket.summary[:50]})
    table(rows, ["id", "intent", "urgency", "queue", "accuracy", "summary"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 Calls the fast text model four times (about 2K tokens). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

**You should now see** vm-01 and vm-04 routed to `📟 on-call` (outage, access + urgent), vm-02 `billing` and vm-03 `plan_change` to the support queue, accuracy mostly 1.0, and an English summary even for the Spanish voicemail.

### What just happened?

That is the whole pipeline a support team would run on its voicemail box: **audio → transcript (with hints) → typed ticket → queue**, with every step measurable. Add Part 6's flags and you know which tickets need a human to listen.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `413` or "file too large" | Over 25 MB (free tier) | 16 kHz mono, FLAC, or chunk with `transcribe_long` |
| Text appears where nobody speaks | Whisper hallucinating over silence/noise | Check loudness before sending; drop segments flagged by `flag_segments` |
| Product names misspelled | Whisper doesn't know your vocabulary | `prompt` with the terms (≤ 224 tokens); keep the list in config |
| `translate` fails on turbo | Only `whisper-large-v3` translates | `GroqTranscriber("whisper-large-v3", translate=True)` |
| Short clips cost more than expected | 10-second minimum per request | Join short clips into one file |
| Timestamps restart at 0 | Chunks transcribed separately | Shift by the chunk start (`transcribe_long`) |
| WER looks bad but the text reads fine | Formatting ("$49" vs "forty nine dollars") | Normalize both texts the same way before scoring |

## ⚠️ Production notes

> ⚠️ **Production note: recordings are personal data.** (Parts 2–8)
> Voices and what callers say are personal data under most privacy laws (in Colombia, Ley 1581 de 2012). Tell callers they are recorded, keep recordings only as long as needed, restrict who can listen, and check your provider's data retention before sending real calls. The course synthesizes its recordings for this reason.

> ⚠️ **Production note: measure STT on your own audio.** (Parts 3–4)
> Published WER (10–12% for these models) is on public benchmarks. Phone lines, accents and background noise change it a lot. Keep 30–50 real, consented recordings with checked transcripts and re-run them whenever you change model, prompt or preprocessing.

## 🏋️ Exercises

1. **(Easy)** Compute the Whisper cost of 500 voicemails of 25 seconds each with turbo and with large-v3. How much does the 10-second minimum add if the voicemails are 4 seconds instead?
2. **(Medium)** Write `normalized_wer(ref, hyp)` that also maps "gómez"→"gomez" (strip accents) and "$49"→"49". Recompute Part 3's table: how much of the WER was formatting?
3. **(Medium)** Translate vm-03's Spanish transcript with the fast text model instead of `whisper-large-v3`. Compare quality, latency and cost for 1,000 calls of 30 s.
4. **(Stretch)** Add a `needs_human` field to each Part 8 ticket: true if any segment is flagged by `flag_segments`, or the customer ID is missing. Print the share of tickets that would reach a person.

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
for model in ("whisper-large-v3-turbo", "whisper-large-v3"):
    print(model, round(500 * whisper_cost(25, model), 4), "USD")   # 0.1389 and 0.3854
print(round(500 * whisper_cost(4) / (500 * 4 / 3600 * 0.04), 1), "x")   # 2.5x: billed as 10 s each
```

**2.**
```python
import unicodedata

def normalized_wer(ref, hyp):
    def clean(text):
        text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
        return re.sub(r"\$(\d+)", r"\1 dollars", text)
    return wer(clean(ref), clean(hyp))

for vm_id, vm in VOICEMAILS.items():
    print(vm_id, wer(vm["script"], transcripts[vm_id].text), normalized_wer(vm["script"], transcripts[vm_id].text))
```
"forty nine" vs "49" still differs: number normalization is a rabbit hole, which is why benchmark papers publish their normalizer.

**3.** `get_model("fast").invoke("Translate to English:\n" + spanish.text).text`. A 30-second call costs about $0.00033 with turbo + a few hundred text tokens, versus $0.00093 with large-v3; quality is usually similar, and you keep the Spanish original.

**4.**
```python
flags = {vm_id: bool(flag_segments(t)) for vm_id, t in hinted.items()}
needs = {vm_id: flags[vm_id] or rule_ticket(t.text).customer_id is None for vm_id, t in hinted.items()}
print(needs, f"{sum(needs.values()) / len(needs):.0%} to a person")
```
</details>

## 🔎 Docs drill (5 min)

On Groq's speech-to-text page, what is the minimum billed length per request, and which `response_format` do you need for word-level timestamps?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Groq speech to text](https://console.groq.com/docs/speech-to-text) and find the section on file limitations / billing.
2. Requests are billed for a **minimum of 10 seconds** of audio.
3. Word timestamps need `response_format="verbose_json"` **and** `timestamp_granularities=["word"]` (add `"segment"` to keep segments).

</details>

## ✅ Checkpoint

- [ ] I can compute the size of a recording and convert to 16 kHz mono before uploading.
- [ ] I transcribed voicemails with Groq Whisper and scored them with WER against the scripts.
- [ ] I used a spelling-hint prompt and measured what it fixed.
- [ ] I compared transcribing and translating a Spanish voicemail.
- [ ] I flagged a hallucinated segment over noise.
- [ ] I transcribed a long recording in overlapping chunks with correct timestamps.
- [ ] I turned transcripts into typed, routed tickets.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open [Groq speech to text](https://console.groq.com/docs/speech-to-text) and [Groq rate limits](https://console.groq.com/docs/rate-limits). Log in [`docs/research-log.md`](../../docs/research-log.md): the Whisper models and prices today, the free-tier audio seconds per hour and per day, whether streaming transcription is now documented, and what your live run of Part 6 returned for pure noise.

## Summary

- Audio size = rate × channels × bits; send 16 kHz mono and chunk anything over 25 MB.
- `verbose_json` gives segments, confidence fields and (with granularities) word timestamps.
- Score transcripts with WER on known scripts, and look at the errors that matter (IDs, names).
- A `prompt` fixes the terms it contains; only `whisper-large-v3` translates.
- Whisper can invent text over noise: check for speech and flag low-confidence segments.
- Audio → transcript → typed ticket → queue is a measurable, everyday pipeline.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 13-03-text-to-speech.ipynb** → [`13-03-text-to-speech.ipynb`](13-03-text-to-speech.ipynb).

[Module 13 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)